In [2]:
%matplotlib qt5

import hyperspy.api as hs
import pyxem as pxm
import numpy as np
import orix
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import matplotlib as mpl
import pickle

from pathlib import Path
from diffpy.structure import Atom, Lattice, Structure
from diffsims.generators.simulation_generator import SimulationGenerator
from dataclasses import dataclass
from sklearn.decomposition import PCA

from orix.quaternion import symmetry, OrientationRegion
from orix.vector import Vector3d
from orix.plot import IPFColorKeyTSL

c:\Users\denis\hyperspy-bundle_latest\Lib\site-packages\hyperspy\misc\utils.py:72: VisibleDeprecationWarning: `_get_block_pattern` has moved to `hyperspy.misc.dask_utils`. It is for internal use only and may be removed in the future.
  warnings.warn(


In [3]:
## Paths used for loading and saving data
path = Path("D:\\Datasets\\aluminium_tilt_series\\SPED_tilt_series")
path_calibrated = Path("D:\\Datasets\\aluminium_tilt_series\\series_calibrated")

path_target = Path("D:\\Datasets\\aluminium_tilt_series\\target_areas")
path_reference = Path("D:\\Datasets\\aluminium_tilt_series\\reference_areas")

path_target_mean = Path("D:\\Datasets\\aluminium_tilt_series\\target_mean")
path_reference_mean = Path("D:\\Datasets\\aluminium_tilt_series\\reference_mean")

path_target_sum = Path("D:\\Datasets\\aluminium_tilt_series\\target_sum")
path_reference_sum = Path("D:\\Datasets\\aluminium_tilt_series\\reference_sum")

path_target_masked = Path("D:\\Datasets\\aluminium_tilt_series\\target_areas_masked")
path_reference_masked = Path("D:\\Datasets\\aluminium_tilt_series\\reference_areas_masked")

path_target_template_disk = Path("D:\\Datasets\\aluminium_tilt_series\\target_template_disk")
path_reference_template_disk = Path("D:\\Datasets\\aluminium_tilt_series\\reference_template_disk")
raw_files = sorted(path.glob("*.hspy"))
calibrated_files = sorted(path_calibrated.glob("*.hspy"))

target_files = sorted(path_target.glob("*.hspy"))
reference_files = sorted(path_reference.glob("*.hspy"))

target_files_mean = sorted(path_target_mean.glob("*.hspy"))
reference_files_mean = sorted(path_reference_mean.glob("*.hspy"))

target_files_sum = sorted(path_target_sum.glob("*.hspy"))
reference_files_sum = sorted(path_reference_sum.glob("*.hspy"))

target_files_masked = sorted(path_target_masked.glob("*.hspy"))
reference_files_masked = sorted(path_reference_masked.glob("*.hspy"))

target_files_template_disk = sorted(path_target_template_disk.glob("*.hspy"))
reference_files_template_disk = sorted(path_reference_template_disk.glob("*.hspy"))

In [18]:
data = hs.load(calibrated_files[43])

In [22]:
ny, nx = data.axes_manager.signal_shape
centers = (ny/2 - 0.5, nx/2 - 0.5)
data.calibration(center=centers)

In [23]:
# Define the crystal structure for aluminum (Al)
a = 4.05
atoms = [Atom('Al', [0, 0, 0]), Atom('Al', [0.5, 0.5, 0]), Atom('Al', [0.5, 0, 0.5]), Atom('Al', [0, 0.5, 0.5])]
lattice = Lattice(a, a, a, 90, 90, 90)
phase = orix.crystal_map.Phase(name='Al', space_group=225, structure=Structure(atoms, lattice))

angular_resolution = 0.5
grid = orix.sampling.get_sample_reduced_fundamental(angular_resolution, point_group=phase.point_group)
orientations = orix.quaternion.Orientation(grid, symmetry=phase.point_group)

simgen = SimulationGenerator(precession_angle=1., minimum_intensity=1e-5)
simulations = simgen.calculate_diffraction2d(phase=phase, rotation=grid, reciprocal_radius=1.5, with_direct_beam=False, max_excitation_error=0.01)

# Create IPF color keys for x, y, and z directions
key_x = IPFColorKeyTSL(phase.point_group, Vector3d.xvector())
key_y = IPFColorKeyTSL(phase.point_group, Vector3d.yvector())
key_z = IPFColorKeyTSL(phase.point_group, Vector3d.zvector())

In [24]:
data.change_dtype("float32")
polar = data.get_azimuthal_integral2d(npt=256)

  0%|          | 0/33 [00:00<?, ?it/s]

In [25]:
result = polar.get_orientation(simulations, n_best=grid.size, frac_keep=0.4)

  0%|          | 0/129 [00:00<?, ?it/s]

WARNING | Hyperspy | The function you applied does not take into account the difference of scales in-between axes. (hyperspy.signal:5597)
WARNING | Hyperspy | The function you applied does not take into account the difference of units in-between axes. (hyperspy.signal:5602)


  0%|          | 0/101 [00:00<?, ?it/s]

In [26]:
data.plot(cmap="viridis", norm="symlog")
data.add_marker(result.to_markers(annotate=True))